# LangSmith: Automatic Tracing & Automated Evaluation

## Core Concepts
* **Tracing (`LANGSMITH_TRACING=true`):** Automatically records a step-by-step execution tree (prompts, responses, latency, tokens) every time your LangChain app runs. No extra code needed.
* **Evaluation (`client.evaluate`):** Tests your app against a saved **Dataset** of questions and grades the answers automatically using **Code Evaluators** (exact rules) and **LLM-as-a-Judge Evaluators** (AI grading AI).

## Section 1: Load Environment Variables & Initialize Gemini + LangSmith Client
* `load_dotenv(override=True)` activates `LANGSMITH_TRACING=true` and loads both API keys.
* Initialize **Gemini (`gemini-3.5-flash-lite`)** and the **LangSmith `Client()`**.

In [1]:
import os
from dotenv import load_dotenv
from langsmith import Client
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

# Load GOOGLE_API_KEY and LANGSMITH_* variables from .env
load_dotenv(override=True)

# Initialize LangSmith client and Gemini model
ls_client = Client()
llm = init_chat_model("google_genai:gemini-3.5-flash-lite", max_retries=5)

print("Tracing enabled:", os.getenv("LANGSMITH_TRACING"))
print("LangSmith Project:", os.getenv("LANGSMITH_PROJECT"))

Tracing enabled: true
LangSmith Project: llm-tracing-and-evals-demo


## Section 2: Automatic Tracing (Zero Extra Code)
Because `LANGSMITH_TRACING=true` is set in `.env`, simply invoking our agent automatically sends the full trace to your LangSmith dashboard under `llm-tracing-and-evals-demo`.
* Run this cell, then open `smith.langchain.com` $\rightarrow$ **Projects** $\rightarrow$ **`llm-tracing-and-evals-demo`** to inspect the exact prompt, output, token count, and latency.

In [2]:
support_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=(
        "You are a concise tech support assistant for CloudBox Storage. "
        "Rules: Keep answers under 3 sentences. "
        "Free plan = 15 GB storage. Pro plan = 2 TB storage for $10/month. "
        "Refunds are allowed within 14 days of purchase."
    ),
)

# Running invoke() automatically logs a trace in LangSmith
trace_test = support_agent.invoke({
    "messages": [{"role": "user", "content": "How much storage do I get on the Pro plan and what does it cost?"}]
})

print("=== SECTION 2: AGENT RESPONSE (CHECK LANGSMITH FOR TRACE) ===")
print(trace_test["messages"][-1].content)

=== SECTION 2: AGENT RESPONSE (CHECK LANGSMITH FOR TRACE) ===
[{'type': 'text', 'text': 'The Pro plan gives you 2 TB of storage for $10 per month.', 'extras': {'signature': 'EpYBCpMBAVjVB3uVKyDh7o05c0Dskdo1V+w4fU+cFHtFv2+NWI/iYRS+oJZ69Af19fTM11EvKZ4HIGnCjUarOo3NuxWX/gSy3z0d686IbyjUe28H4eHwb23nfCK7TdAZOUSU2Wm1IPAdZZVpAoV2luLgjNzhddsjJNDlz7Vl57huwJn1dUG6LhP2FntLoZ0pj5d5UWqLxoIj'}}]


## Section 3: Create a Test Dataset (`ls_client.create_dataset`)
To run evaluations, we first create a **Dataset** of test questions (`inputs`) and ground-truth reference answers (`outputs`).
* Once saved in LangSmith, you can reuse this exact dataset every time you edit your prompt or switch models.

In [3]:
dataset_name = "cloudbox-support-qa-v1"

# Create dataset only if it doesn't already exist
if not ls_client.has_dataset(dataset_name=dataset_name):
    dataset = ls_client.create_dataset(
        dataset_name=dataset_name,
        description="Test questions and ground-truth answers for CloudBox support agent."
    )

    examples = [
        {
            "inputs": {"question": "How much free storage do I get?"},
            "outputs": {"answer": "The Free plan includes 15 GB of storage."}
        },
        {
            "inputs": {"question": "What is the price and storage limit of the Pro plan?"},
            "outputs": {"answer": "The Pro plan offers 2 TB of storage for $10 per month."}
        },
        {
            "inputs": {"question": "Can I get a refund after 20 days?"},
            "outputs": {"answer": "No, refunds are only allowed within 14 days of purchase."}
        },
        {
            "inputs": {"question": "Do you offer an Enterprise unlimited plan?"},
            "outputs": {"answer": "Only Free (15 GB) and Pro (2 TB for $10/month) plans are mentioned; no Enterprise plan is listed."}
        },
    ]

    ls_client.create_examples(dataset_id=dataset.id, examples=examples)
    print(f"Created dataset '{dataset_name}' with {len(examples)} test examples!")
else:
    print(f"Dataset '{dataset_name}' already exists and is ready!")

Dataset 'cloudbox-support-qa-v1' already exists and is ready!


## Section 4: Define the Target Function & Two Types of Evaluators
An evaluation experiment needs two things:
1. **Target Function (`run_support_agent`):** Takes a dataset input (`{"question": ...}`) and returns the agent's answer (`{"answer": ...}`).
2. **Evaluators (Graders):**
   * **Evaluator 1 — Code Rule (`concise_length_check`):** Pure Python check returning `1` if the answer is concise ($\le 350$ characters) and `0` otherwise.
   * **Evaluator 2 — LLM-as-a-Judge (`correctness_judge`):** Uses Gemini to compare the agent's answer against the reference answer and score `1` (correct) or `0` (incorrect/hallucinated).

In [4]:
def get_message_text(message) -> str:
    if hasattr(message, "text") and isinstance(message.text, str):
        return message.text.strip()
    content = message.content
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        return "".join(
            block.get("text", "") if isinstance(block, dict) else str(block)
            for block in content
        ).strip()
    return str(content).strip()

# 1. Target function: wraps our agent so LangSmith can feed dataset inputs into it
def run_support_agent(inputs: dict) -> dict:
    response = support_agent.invoke({
        "messages": [{"role": "user", "content": inputs["question"]}]
    })
    return {"answer": get_message_text(response["messages"][-1])}

# 2. Evaluator 1: Deterministic Python Code Check (length rule)
def concise_length_check(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    answer = outputs.get("answer", "")
    is_concise = len(answer) <= 350
    return {
        "key": "concise_length",
        "score": 1 if is_concise else 0,
    }

# 3. Evaluator 2: LLM-as-a-Judge (factual correctness against reference answer)
judge_llm = init_chat_model("google_genai:gemini-3.5-flash-lite", max_retries=5)

def correctness_judge(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    prompt = f"""You are a strict grader. Compare the Student Answer to the Reference Answer.
Reply with ONLY '1' if the Student Answer is factually accurate and matches the Reference Answer, or '0' if it is wrong or hallucinates extra facts.

Question: {inputs['question']}
Reference Answer: {reference_outputs['answer']}
Student Answer: {outputs['answer']}"""

    judge_msg = judge_llm.invoke(prompt)
    verdict = get_message_text(judge_msg)
    score = 1 if "1" in verdict else 0
    return {
        "key": "correctness",
        "score": score,
    }

print("Target function and both evaluators (Code + LLM-as-a-Judge) defined!")

Target function and both evaluators (Code + LLM-as-a-Judge) defined!


## Section 5: Run the Evaluation Experiment & View Results
Calling `ls_client.evaluate(...)` runs all 4 test questions through `run_support_agent`, scores every output with both evaluators, and uploads the full report card to LangSmith under **Datasets & Experiments**.

In [5]:
experiment_results = ls_client.evaluate(
    run_support_agent,
    data=dataset_name,
    evaluators=[concise_length_check, correctness_judge],
    experiment_prefix="cloudbox-baseline-v1",
)

print("=== EVALUATION COMPLETE ===")
print("Experiment Name:", experiment_results.experiment_name)

View the evaluation results for experiment: 'cloudbox-baseline-v1-3abe32cc' at:
https://smith.langchain.com/o/dfbd7f39-0e01-469a-b87e-4f7f944b21fd/datasets/e90ae325-9f56-4110-aee4-2c503d0b1555/compare?selectedSessions=7dfae8b3-badb-4139-8292-4b185291eb75


=== EVALUATION COMPLETE ===
Experiment Name: cloudbox-baseline-v1-3abe32cc


## Section 6: Inspect Each Question's Score Right Inside the Notebook
Besides viewing the visual dashboard in LangSmith, we can iterate over `experiment_results` directly in Python to inspect every question's answer and scores.

In [6]:
for row in experiment_results:
    question = row["example"].inputs["question"]
    expected = row["example"].outputs["answer"]
    actual = row["run"].outputs["answer"]
    scores = {res.key: res.score for res in row["evaluation_results"]["results"]}

    print(f"Q: {question}")
    print(f"Expected: {expected}")
    print(f"Actual:   {actual}")
    print(f"Scores:   {scores}")
    print("-" * 60)

Q: Can I get a refund after 20 days?
Expected: No, refunds are only allowed within 14 days of purchase.
Actual:   No, refunds are only allowed within 14 days of purchase. Therefore, a refund after 20 days is not eligible.
Scores:   {'concise_length': 1, 'correctness': 1}
------------------------------------------------------------
Q: Do you offer an Enterprise unlimited plan?
Expected: Only Free (15 GB) and Pro (2 TB for $10/month) plans are mentioned; no Enterprise plan is listed.
Actual:   No, CloudBox Storage does not currently offer an unlimited enterprise plan. Our largest offering is the Pro plan, which provides 2 TB of storage for $10/month.
Scores:   {'concise_length': 1, 'correctness': 1}
------------------------------------------------------------
Q: What is the price and storage limit of the Pro plan?
Expected: The Pro plan offers 2 TB of storage for $10 per month.
Actual:   The Pro plan offers 2 TB of storage for $10 per month.
Scores:   {'concise_length': 1, 'correctness':

## Summary: Tracing vs. Evaluation Workflow

| Step | What You Used | What Happened |
| :--- | :--- | :--- |
| **1. Automatic Tracing** | `LANGSMITH_TRACING=true` in `.env` | Every `agent.invoke()` call logged its full execution tree automatically |
| **2. Dataset Creation** | `ls_client.create_dataset()` | Saved 4 standardized test inputs + reference answers in LangSmith |
| **3. Code Evaluator** | `concise_length_check()` | Graded output length deterministically (`0` or `1`) using Python |
| **4. LLM-as-a-Judge** | `correctness_judge()` | Used a second Gemini call to grade factual accuracy against reference answers |
| **5. Experiment Run** | `ls_client.evaluate()` | Scored the whole dataset and saved a comparison report in LangSmith |